In [ ]:
# Cell 1: Setup, Imports, Reproducibility
import os
import json
import copy
import random
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

sns.set_style("darkgrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Kaggle Data Path and Fixed Class Mapping
# Updated with your exact dataset path
DATA_DIR = Path("/kaggle/input/datasets/fadyyoussif/dataset/Balanced Ham10000 Dataset")
MODEL_OUT = Path("/kaggle/working/skin_resnet50.pth")
CHECKPOINT_OUT = Path("/kaggle/working/skin_resnet50_checkpoint.pth")
MAPPING_OUT = Path("/kaggle/working/class_mapping.json")
METRICS_OUT = Path("/kaggle/working/training_metrics.json")

DX_TO_NAME = {
    "akiec": "Actinic Keratoses",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis-like Lesions",
    "df": "Dermatofibroma",
    "nv": "Melanocytic Nevi (Moles)",
    "mel": "Melanoma",
    "vasc": "Vascular Lesions",
}

LABEL_MAP = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "df": 3,
    "nv": 4,
    "mel": 5,
    "vasc": 6,
}

CLASS_NAMES = [
    "Actinic Keratoses",
    "Basal Cell Carcinoma",
    "Benign Keratosis-like Lesions",
    "Dermatofibroma",
    "Melanocytic Nevi (Moles)",
    "Melanoma",
    "Vascular Lesions",
]

assert len(CLASS_NAMES) == 7
assert set(LABEL_MAP.values()) == set(range(7))

print("Fixed class order used by training and API:")
for dx, idx in sorted(LABEL_MAP.items(), key=lambda x: x[1]):
    print(f"{idx}: {dx:6s} -> {DX_TO_NAME[dx]}")

In [ ]:
# Cell 3: Ultimate Robust Directory Traversal
data = []
valid_extensions = {'.jpg', '.jpeg', '.png'}

# 1. Check if directory exists
if not DATA_DIR.exists():
    raise FileNotFoundError(f"المسار ده مش موجود أصلاً: {DATA_DIR}")

# 2. Recursively find all images
print(f"Scanning directories inside: {DATA_DIR} ...")
all_images = list(DATA_DIR.rglob("*.*"))

for img_path in all_images:
    if img_path.suffix.lower() in valid_extensions:
        # هندور في كل أجزاء المسار، مش بس الفولدر الأب المباشر
        # عشان لو الصور جوه فولدر فرعي أو المسار فيه مسافات
        dx_found = None
        for part in img_path.parts:
            clean_part = part.lower().strip()
            if clean_part in LABEL_MAP:
                dx_found = clean_part
                break
        
        if dx_found:
            data.append({
                "path": str(img_path),
                "class_name": DX_TO_NAME[dx_found],
                "label": LABEL_MAP[dx_found],
                "dx": dx_found
            })

df = pd.DataFrame(data)

# 3. Guard against empty DataFrame with Detailed Debugging
if df.empty:
    print("\n--- 🛑 DEBUG INFO ---")
    print(f"Total files found (any type) inside path: {len(all_images)}")
    if len(all_images) > 0:
        print("أول 5 ملفات الكود شافها (عشان نعرف المشكلة في المسار ولا الامتداد):")
        for f in all_images[:5]:
            print(f)
    else:
        print("الفولدر موجود بس فاضي تماماً مفيش جواه أي ملفات! راجع مسار الداتا في Kaggle.")
    raise ValueError("الـ Pipeline فشل يلاقي صور متوافقة. راجع الـ DEBUG INFO فوق عشان تعرف الكود شايف الداتا إزاي.")

print(f"\nTotal images found and mapped successfully: {len(df)}")

print("\nBalanced class distribution:")
print(df.groupby(["label", "class_name"]).size())

plt.figure(figsize=(12, 5))
sns.countplot(data=df, y="class_name", order=df["class_name"].value_counts().index, palette="viridis")
plt.title("Balanced HAM10000 Class Distribution")
plt.xlabel("Count")
plt.ylabel("Class")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Stratified Train/Validation Split
# Ensuring both Train and Val sets inherit the balanced distribution
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

print("\nTrain distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

In [ ]:
# Cell 6: Dataset, Transforms, and DataLoaders
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0), ratio=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        if self.transform:
            image = self.transform(image)
        return image, label

BATCH_SIZE = 32
NUM_WORKERS = 2

train_ds = SkinDataset(train_df, transform=train_transform)
val_ds = SkinDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,  # Standard uniform shuffling for balanced data
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Cell 7: Quick Batch Sanity Check
images, labels = next(iter(train_loader))
print("Batch image tensor:", images.shape)
print("Batch label tensor:", labels.shape)
print("Batch labels:", labels[:16].tolist())

plt.figure(figsize=(12, 6))
for i in range(min(8, images.size(0))):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(CLASS_NAMES[int(labels[i])], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: ResNet50 Model Setup with Two-Stage Fine-Tuning & Updated Hyperparameters
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(model.fc.in_features, 7),
)

# Stage 1: Freeze backbone, train only the classifier head.
for name, param in model.named_parameters():
    param.requires_grad = name.startswith("fc")

model = model.to(device)

# Standard cross entropy with label smoothing (No class weights needed)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

# Updated Epoch configuration and Early Stopping Patience
STAGE1_EPOCHS = 6
STAGE2_EPOCHS = 24
TOTAL_EPOCHS = STAGE1_EPOCHS + STAGE2_EPOCHS
PATIENCE = 10  # Set early stopping patience to 10 epochs

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params stage 1: {trainable_params:,} / {total_params:,}")
print(f"Early Stopping Patience set to: {PATIENCE}")

In [ ]:
# Cell 9: Training Execution Helpers
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_balanced_acc": [],
    "val_weighted_f1": [],
    "lr": [],
}

best_metric = -1.0
best_epoch = 0
best_weights = copy.deepcopy(model.state_dict())
patience_counter = 0

def run_train_epoch():
    model.train()
    total_loss = 0.0
    y_true, y_pred = [], []

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        y_true.extend(labels.detach().cpu().numpy())
        y_pred.extend(outputs.detach().argmax(1).cpu().numpy())

    return total_loss / len(train_loader), accuracy_score(y_true, y_pred)

def run_validation_epoch():
    model.eval()
    total_loss = 0.0
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            probs = F.softmax(outputs, dim=1)

            total_loss += loss.item()
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(outputs.argmax(1).cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    metrics = {
        "loss": total_loss / len(val_loader),
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }
    return metrics

In [ ]:
# Cell 10: Two-Stage Training Loop Engine
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS, eta_min=1e-6)
print("Starting training optimization workflow...")

for epoch in range(1, TOTAL_EPOCHS + 1):
    if epoch == STAGE1_EPOCHS + 1:
        print("\n[Stage 2] Unfreezing Layer3, Layer4, and FC for full deep fine-tuning.\n")
        for name, param in model.named_parameters():
            param.requires_grad = name.startswith("layer3") or name.startswith("layer4") or name.startswith("fc")
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS, eta_min=1e-6)
        print(f"Trainable params stage 2: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    train_loss, train_acc_value = run_train_epoch()
    val_metrics = run_validation_epoch()
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc_value)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])
    history["val_balanced_acc"].append(val_metrics["balanced_acc"])
    history["val_weighted_f1"].append(val_metrics["weighted_f1"])
    history["lr"].append(current_lr)

    score = val_metrics["weighted_f1"]
    improved = score > best_metric
    if improved:
        best_metric = score
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save(best_weights, MODEL_OUT)
        torch.save(
            {
                "model_state_dict": best_weights,
                "class_names": CLASS_NAMES,
                "label_map": LABEL_MAP,
                "dx_to_name": DX_TO_NAME,
                "best_epoch": best_epoch,
                "best_weighted_f1": float(best_metric),
                "val_accuracy": float(val_metrics["acc"]),
                "val_balanced_accuracy": float(val_metrics["balanced_acc"]),
            },
            CHECKPOINT_OUT,
        )
    else:
        patience_counter += 1

    marker = "⭐ BEST" if improved else ""
    print(
        f"Epoch {epoch:02d}/{TOTAL_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc_value*100:.2f}% | "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']*100:.2f}% | "
        f"bal_acc={val_metrics['balanced_acc']*100:.2f}% wf1={val_metrics['weighted_f1']:.4f} | "
        f"lr={current_lr:.2e} {marker}"
    )

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping triggered at epoch {epoch}. No progress for {PATIENCE} consecutive epochs.")
        print(f"Best Weights belong to Epoch {best_epoch} with Weighted F1: {best_metric:.4f}")
        break

model.load_state_dict(best_weights)
print(f"\nSuccessfully restored optimal weights from epoch {best_epoch}.")

In [ ]:
# Cell 11: Performance Curves Tracking
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs, history["val_loss"], label="Val Loss", marker="o")
axes[0].set_title("Loss Trajectory")
axes[0].legend()

axes[1].plot(epochs, np.array(history["train_acc"]) * 100, label="Train Acc", marker="o")
axes[1].plot(epochs, np.array(history["val_acc"]) * 100, label="Val Acc", marker="o")
axes[1].plot(epochs, np.array(history["val_balanced_acc"]) * 100, label="Val Balanced Acc", marker="o")
axes[1].set_title("Accuracy Tracking")
axes[1].legend()

axes[2].plot(epochs, history["val_weighted_f1"], label="Val Weighted F1", marker="o")
axes[2].set_title("Weighted F1 Score Evolution")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal Model Saved at Epoch: {best_epoch}")
print(f"Maximized Validation Weighted F1: {best_metric:.4f}")

In [ ]:
# Cell 12: Comprehensive Classification Report & Confusion Matrix Analysis
final_metrics = run_validation_epoch()
y_true = final_metrics["y_true"]
y_pred = final_metrics["y_pred"]
y_prob = final_metrics["y_prob"]

print(f"Final Validation Accuracy: {final_metrics['acc']*100:.2f}%")
print(f"Final Validation Balanced Accuracy: {final_metrics['balanced_acc']*100:.2f}%")
print(f"Final Validation Weighted F1: {final_metrics['weighted_f1']:.4f}")
print(f"Final Validation Macro F1: {final_metrics['macro_f1']:.4f}")

print("\nDetailed Multi-Class Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=list(range(7)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix - Optimal Model Test")
plt.xlabel("Predicted Class Mapping")
plt.ylabel("Actual Ground Truth")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

pred_pct = pd.Series(y_pred).value_counts(normalize=True).reindex(range(7), fill_value=0) * 100
bias_table = pd.DataFrame({
    "class_index": range(7),
    "class_name": CLASS_NAMES,
    "prediction_percent": pred_pct.values,
})
display(bias_table)

In [ ]:
# Cell 13: Multi-Class ROC AUC Profile
classes = list(range(7))
y_true_bin = label_binarize(y_true, classes=classes)

print("Computing ROC AUC performance indexes per class:")
roc_auc = {}
fpr = {}
tpr = {}
for i in classes:
    try:
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        print(f"Class {i}: {CLASS_NAMES[i]:35s} AUC = {roc_auc[i]:.4f}")
    except ValueError:
        roc_auc[i] = np.nan
        print(f"Class {i}: {CLASS_NAMES[i]:35s} AUC = N/A")

try:
    macro_auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class=\"ovr\")
    weighted_auc = roc_auc_score(y_true_bin, y_prob, average=\"weighted\", multi_class=\"ovr\")
    print(f"\nGlobal Macro AUC: {macro_auc:.4f}")
    print(f"Global Weighted AUC: {weighted_auc:.4f}")
except ValueError as exc:
    print(f"AUC Statistics Summary is currently unavailable: {exc}")

plt.figure(figsize=(12, 8))
for i in classes:
    if not np.isnan(roc_auc[i]):
        plt.plot(fpr[i], tpr[i], lw=2, label=f"{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=2)
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("Receiver Operating Characteristic (ROC) Subplots")
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Visual Predictive Test Verification on Out-of-Sample Split
model.eval()
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for class_idx in range(7):
    sample = val_df[val_df["label"] == class_idx].sample(1, random_state=SEED).iloc[0]
    img = Image.open(sample["path"]).convert("RGB")
    tensor = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    axes[class_idx].imshow(img)
    axes[class_idx].set_title(
        f"True: {CLASS_NAMES[class_idx]}\nPred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]*100:.1f}%)",
        fontsize=8,
    )
    axes[class_idx].axis("off")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: Artifact File Serializer & Local File System Generation
MAPPING_OUT.write_text(json.dumps({str(i): name for i, name in enumerate(CLASS_NAMES)}, indent=2), encoding="utf-8")

metrics_to_save = {
    "best_epoch": int(best_epoch),
    "best_weighted_f1": float(best_metric),
    "val_accuracy": float(final_metrics["acc"]),
    "val_balanced_accuracy": float(final_metrics["balanced_acc"]),
    "val_weighted_f1": float(final_metrics["weighted_f1"]),
    "val_macro_f1": float(final_metrics["macro_f1"]),
    "prediction_percent_by_class": {CLASS_NAMES[i]: float(pred_pct.loc[i]) for i in range(7)},
}
METRICS_OUT.write_text(json.dumps(metrics_to_save, indent=2), encoding="utf-8")

print(f"Artifact Serialized Successfully:")
print(f"-> Model State Dict: {MODEL_OUT}")
print(f"-> Heavyweight Checkpoint: {CHECKPOINT_OUT}")
print(f"-> Class Map Lookup: {MAPPING_OUT}")
print(f"-> Structural JSON Metrics: {METRICS_OUT}\n")

from IPython.display import FileLink, display

display(FileLink(str(MODEL_OUT)))
display(FileLink(str(CHECKPOINT_OUT)))
display(FileLink(str(MAPPING_OUT)))
display(FileLink(str(METRICS_OUT)))